# Rotary Inverted Pendulum — velocity PID

Host-side PID controller for `PendulumController.ino`.

Improvements: seconds-based PID timing, signed velocity commands, angle wrapping, derivative filtering, anti-windup, output/slew limits, pendulum and rotor guards, and hard-stop-on-exit.

**Safety:** start near the rotor center with the pendulum close to upright. Keep clear of the mechanism and be ready to remove motor power.


In [ ]:
from control_comms import ControlComms, StatusCode, DebugLevel
import time
import matplotlib.pyplot as plt


In [ ]:
SERIAL_PORT = "COM6"  # e.g. /dev/ttyACM0 on Linux
BAUD_RATE = 500000
TIMEOUT = 0.20

CMD_SET_HOME = 0
CMD_MOVE_TO = 1
CMD_MOVE_BY = 2
CMD_SET_STEP_MODE = 3
CMD_SET_VELOCITY = 4
CMD_HARD_STOP = 5
CMD_QUERY = 6
CMD_RESET_SAFETY = 7
STEP_MODE_16 = 4

STATUS_OK = 0
STATUS_MOVING = 1
STATUS_LIMIT = 2
STATUS_DRIVER_FAULT = 3

SETPOINT_DEG = 180.0
KP, KI, KD = 35.0, 0.0, 1.2
CONTROL_SIGN = 1.0
MAX_SPEED_PPS = 1600.0
MAX_DV_PPS_PER_S = 12000.0
INTEGRAL_LIMIT = 20.0
DERIVATIVE_TAU_S = 0.020
PENDULUM_GUARD_DEG = 35.0
ROTOR_GUARD_DEG = 80.0
CONTROL_TIME_S = 15.0
MAX_SAMPLES = 10000


In [ ]:
def wrap_deg(angle):
    return (angle + 180.0) % 360.0 - 180.0

def clamp(x, lo, hi):
    return max(lo, min(hi, x))

def slew_limit(target, previous, max_rate, dt):
    d = max_rate * dt
    return clamp(target, previous - d, previous + d)

def unpack(resp):
    if resp is None:
        raise RuntimeError("No response from controller")
    status, timestamp_ms, terminated, obs = resp
    if len(obs) < 4:
        raise RuntimeError(f"Expected 4 observations, got {len(obs)}")
    return status, timestamp_ms, terminated, *obs[:4]


In [ ]:
ctrl = ControlComms(timeout=TIMEOUT, debug_level=DebugLevel.DEBUG_ERROR)
if ctrl.connect(SERIAL_PORT, BAUD_RATE) is not StatusCode.OK:
    raise RuntimeError(f"Could not connect to {SERIAL_PORT}")
time.sleep(0.25)
print("Connected:", SERIAL_PORT)
print("Query:", unpack(ctrl.step(CMD_QUERY, [0.0])))
print("Reset:", unpack(ctrl.step(CMD_RESET_SAFETY, [0.0])))
print("Step mode:", unpack(ctrl.step(CMD_SET_STEP_MODE, [STEP_MODE_16])))
print("Home:", unpack(ctrl.step(CMD_SET_HOME, [0.0])))


## Low-speed direction test

Run before PID. If the physical feedback direction is wrong, change `CONTROL_SIGN` to `-1.0`.


In [ ]:
try:
    print(unpack(ctrl.step(CMD_SET_VELOCITY, [250.0])))
    time.sleep(0.20)
finally:
    print(unpack(ctrl.step(CMD_HARD_STOP, [0.0])))


In [ ]:
log = {k: [] for k in (
    "t", "pend_deg", "rotor_deg", "error_deg",
    "command_pps", "measured_pps", "status"
)}

integral = 0.0
derivative_f = 0.0
previous_error = None
previous_command = 0.0
previous_timestamp_ms = None
start_host = time.monotonic()

try:
    resp = ctrl.step(CMD_SET_VELOCITY, [0.0])

    for _ in range(MAX_SAMPLES):
        status, timestamp_ms, terminated, pend_deg, rotor_deg, motor_pps, drv = unpack(resp)
        error = wrap_deg(SETPOINT_DEG - pend_deg)

        if terminated or status in (STATUS_LIMIT, STATUS_DRIVER_FAULT):
            print(f"Firmware safety stop: status={status}, driver=0x{int(drv):04X}")
            break
        if abs(error) > PENDULUM_GUARD_DEG:
            print(f"Pendulum guard: error={error:.2f} deg")
            break
        if abs(rotor_deg) > ROTOR_GUARD_DEG:
            print(f"Rotor guard: rotor={rotor_deg:.2f} deg")
            break

        if previous_timestamp_ms is None:
            dt = 0.005
        else:
            delta_ms = (timestamp_ms - previous_timestamp_ms) & 0xFFFFFFFF
            dt = clamp(delta_ms * 1e-3, 0.0005, 0.050)
        previous_timestamp_ms = timestamp_ms

        raw_d = 0.0 if previous_error is None else (error - previous_error) / dt
        previous_error = error
        alpha = dt / (DERIVATIVE_TAU_S + dt)
        derivative_f += alpha * (raw_d - derivative_f)

        candidate_i = clamp(integral + error * dt, -INTEGRAL_LIMIT, INTEGRAL_LIMIT)
        u_candidate = CONTROL_SIGN * (KP * error + KI * candidate_i + KD * derivative_f)
        pushes_high = u_candidate > MAX_SPEED_PPS and error * CONTROL_SIGN > 0
        pushes_low = u_candidate < -MAX_SPEED_PPS and error * CONTROL_SIGN < 0
        if not (pushes_high or pushes_low):
            integral = candidate_i

        command = CONTROL_SIGN * (KP * error + KI * integral + KD * derivative_f)
        command = clamp(command, -MAX_SPEED_PPS, MAX_SPEED_PPS)
        command = slew_limit(command, previous_command, MAX_DV_PPS_PER_S, dt)
        previous_command = command

        t = time.monotonic() - start_host
        values = (t, pend_deg, rotor_deg, error, command, motor_pps, status)
        for key, value in zip(log, values):
            log[key].append(value)

        if t >= CONTROL_TIME_S:
            print("Control time complete.")
            break

        resp = ctrl.step(CMD_SET_VELOCITY, [command])
        if resp is None:
            print("Serial timeout; stopping.")
            break

except KeyboardInterrupt:
    print("Interrupted.")
finally:
    try:
        ctrl.step(CMD_HARD_STOP, [0.0])
    except Exception:
        pass
    print("Motor stop requested.")

print("samples:", len(log["t"]))


In [ ]:
if log["t"]:
    plt.figure(figsize=(10, 4))
    plt.plot(log["t"], log["error_deg"], label="pendulum error [deg]")
    plt.plot(log["t"], log["rotor_deg"], label="rotor angle [deg]")
    plt.xlabel("time [s]")
    plt.grid(True)
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(log["t"], log["command_pps"], label="command [pps]")
    plt.plot(log["t"], log["measured_pps"], label="measured [pps]")
    plt.xlabel("time [s]")
    plt.grid(True)
    plt.legend()
    plt.show()


In [ ]:
try:
    ctrl.step(CMD_HARD_STOP, [0.0])
finally:
    ctrl.close()
print("Closed.")
